In [1]:
import os
import numpy as np
import numpy as np
# import torch
# from arsf_envi_reader import envi_header
import shutil
import os
import json
import math
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from osgeo import gdal,ogr,osr
from scipy.optimize import curve_fit
# from tqdm import tqdm
# import multiprocess as mp
from scipy import ndimage
from numpy import trapz
from scipy import stats
import rasterio

In [2]:
npy_folder = r'E:\wenqu\numpy_file\npy_file\site2c'  # where .npy files are stored
reference_tif = r'D:\wenqu\chapter1_2\updated_uas_modeling\uas_trait_map\site2c_pc.tif'    # an existing .tif file to copy georeferencing info


In [3]:
ref_ds = gdal.Open(reference_tif)
width = ref_ds.RasterXSize
height = ref_ds.RasterYSize
print(width, height)

4744 5470


# Graminoid

In [4]:
output_tif = r'D:\wenqu\chapter1_2\chapter2\PFT_ln_cover\No_scale\site2c_graminoid.tif'


# Your list of selected bands
band_names = ['b60_mean', 'b85_std', 'b64_std', 'b113_mean', 'b121_mean',
       'b43_std', 'b62_std', 'b82_std', 'b99_std', 'b11_std', 'b84_std',
       'b98_mean', 'b50_std', 'b63_std', 'b26_mean', 'b15_mean',
       'b122_mean', 'b6_mean', 'b65_std', 'b28_std', 'b8_mean',
       'b118_mean', 'b22_std', 'b17_std', 'b17_mean', 'b95_std',
       'b117_mean', 'b73_std', 'b66_std', 'b11_mean', 'b103_std',
       'b7_mean', 'b16_mean', 'b107_mean', 'b2_mean', 'b26_std',
       'b104_std', 'b30_std', 'b4_mean', 'b16_std', 'b21_mean',
       'b115_std', 'b4_std', 'b5_std', 'b122_std', 'b15_std', 'b13_std',
       'b10_std', 'b119_std', 'b3_std', 'b110_std', 'b120_std', 'b6_std',
       'b107_std', 'b118_std']


plsr_coefficients  = np.array([58.0317442319411,
 36.951644723481216,
 -27.66102956057662,
 -19.26142547484908,
 8.916751339816052,
 224.65140786105331,
 -114.15275001613202,
 24.146508192857297,
 -40.044109115594935,
 92.45015500152975,
 24.178660362629817,
 -12.658272365642873,
 74.56878170341204,
 -12.651810342276526,
 213.41776076987864,
 -128.42069688703387,
 15.72426352206476,
 93.72833796527988,
 -30.86501624720157,
 -55.79147175386639,
 11.014390763573944,
 -0.28280984358364136,
 -144.31935907750062,
 -388.8623382561275,
 -109.17584598862476,
 -36.4113203265246,
 -18.49566342895585,
 30.65339970353203,
 -31.993971553613047,
 133.61084988234967,
 -7.297630412446527,
 334.8424886612602,
 -127.32448203391445,
 11.873129614491633,
 46.26763381779648,
 243.30607102958277,
 -48.49823829180852,
 -146.3908036393721,
 -154.68549338741528,
 -144.49698260362348,
 -219.7572081755742,
 31.99263587019205,
 -229.04924894358038,
 -304.5819671020216,
 36.810990008009185,
 -130.19125750411945,
 433.5643998777482,
 308.3222812607521,
 -96.09137957177023,
 -240.82656080960766,
 54.46838437636951,
 77.82996676741888,
 590.4551123073679,
 111.62061643987788,
 -126.0985318460854])

intercept = -2.14186386
trait_map = None


# Initialize trait map
trait_map = None
first_band = True

# Load reference image for size and geo info
ref_ds = gdal.Open(reference_tif)
geotransform = ref_ds.GetGeoTransform()
projection = ref_ds.GetProjection()
cols = ref_ds.RasterXSize
rows = ref_ds.RasterYSize
ref_ds = None

# Initialize trait map and background mask
trait_map = np.zeros((rows, cols), dtype=np.float32)
background_mask = None

# Process each band
for idx, (band_name, coef) in enumerate(zip(band_names, plsr_coefficients)):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    
    if not os.path.exists(band_path):
        print(f"Warning: Missing band file {band_path}")
        continue
    
    try:
        band_array = np.load(band_path)
        
        # Check dimension match
        if band_array.shape != (rows, cols):
            print(f"Warning: {band_name} has wrong dimensions {band_array.shape}, expected {(rows, cols)}")
            continue
        
        # Create background mask from the FIRST band
        if idx == 0:
            background_mask = (band_array == 0)  # True = background
        
        # Multiply only valid data
        trait_map += band_array * coef
    
    except Exception as e:
        print(f"Error processing {band_name}: {str(e)}")
        continue

# Add intercept
trait_map += intercept

# Mask background as NaN
trait_map = np.where(background_mask, 0, trait_map)

# Write output GeoTIFF
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(output_tif, cols, rows, 1, gdal.GDT_Float32)
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)
out_ds.GetRasterBand(1).WriteArray(trait_map)
out_ds.FlushCache()
out_ds = None

print(f"Successfully created trait map: {output_tif}")
print(f"Trait map stats - Min: {np.nanmin(trait_map):.2f}, Max: {np.nanmax(trait_map):.2f}, Mean: {np.nanmean(trait_map):.2f}")


Successfully created trait map: D:\wenqu\chapter1_2\chapter2\PFT_ln_cover\No_scale\site2c_graminoid.tif
Trait map stats - Min: -15.64, Max: 11.16, Mean: -0.82


# lichen

# scale

In [5]:
output_tif = r'D:\wenqu\chapter1_2\chapter2\PFT_ln_cover\No_scale\site2c_lichen.tif'


# Your list of selected bands
band_names = ['b83_std', 'b108_std', 'b25_std', 'b101_mean', 'b58_mean',
       'b34_std', 'b118_mean', 'b82_std', 'b9_mean', 'b29_std', 'b59_std',
       'b28_std', 'b96_std', 'b91_std', 'b35_std', 'b55_std', 'b33_std',
       'b13_mean', 'b109_std', 'b98_std', 'b118_std', 'b107_std',
       'b65_std', 'b48_std', 'b100_mean', 'b5_std', 'b75_std', 'b3_std',
       'b115_mean', 'b7_std', 'b19_std', 'b16_std', 'b31_std', 'b63_std',
       'b112_mean', 'b8_std', 'b105_std', 'b101_std', 'b21_std',
       'b32_std', 'b8_mean', 'b30_std', 'b111_mean', 'b109_mean',
       'b61_std', 'b62_std', 'b120_std', 'b12_std', 'b64_std', 'b56_std',
       'b6_std', 'b9_std', 'b2_mean', 'b10_std', 'b60_std', 'b119_std',
       'b4_std', 'b111_std', 'b122_mean', 'b2_std', 'b1_std', 'b122_std']


plsr_coefficients  = np.array([14.101830474321268,
 13.268872394950563,
 -55.56769625896137,
 -5.607725265592757,
 -21.621290563076077,
 29.720645476778515,
 -2.8642460048012164,
 14.31834947626339,
 36.26855174977243,
 26.005857815243985,
 -24.919945579992348,
 17.094965095148567,
 13.91846335034344,
 15.059716234310757,
 36.56670543299649,
 39.53938455906832,
 21.143470257959237,
 27.669483535309688,
 -2.3422519548527942,
 -2.2294730080783247,
 15.254052233729253,
 -6.228393749324763,
 -14.22743206687467,
 -16.98085810613815,
 -6.091890421033427,
 100.59086646284173,
 16.212722518935998,
 -31.559968569370543,
 1.9145291167480323,
 100.23986490387037,
 -42.054998994678165,
 -63.27751374350366,
 30.760790256337213,
 -19.985791629752388,
 3.163578820492871,
 37.78930308281953,
 -6.499981709839,
 -6.738495379988435,
 65.13543123091563,
 33.12832481241849,
 56.99141279339163,
 32.854622287268846,
 2.195432753772952,
 4.598174664291989,
 -32.375500831055554,
 -28.80167609995413,
 -17.23651444432207,
 -139.33294471991084,
 -20.867725390852918,
 55.19149625708606,
 -72.56257011909179,
 90.01367489180062,
 -23.218890624907413,
 140.1559640946501,
 -47.230879945688294,
 -17.346152161011137,
 175.3192231434232,
 -21.719448069715725,
 -6.943354484145348,
 -92.2543276809186,
 49.02953618547655,
 -41.98475618641193])

intercept = -1.420041
trait_map = None


# Initialize trait map
trait_map = None
first_band = True

# Load reference image for size and geo info
ref_ds = gdal.Open(reference_tif)
geotransform = ref_ds.GetGeoTransform()
projection = ref_ds.GetProjection()
cols = ref_ds.RasterXSize
rows = ref_ds.RasterYSize
ref_ds = None

# Initialize trait map and background mask
trait_map = np.zeros((rows, cols), dtype=np.float32)
background_mask = None

# Process each band
for idx, (band_name, coef) in enumerate(zip(band_names, plsr_coefficients)):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    
    if not os.path.exists(band_path):
        print(f"Warning: Missing band file {band_path}")
        continue
    
    try:
        band_array = np.load(band_path)
        
        # Check dimension match
        if band_array.shape != (rows, cols):
            print(f"Warning: {band_name} has wrong dimensions {band_array.shape}, expected {(rows, cols)}")
            continue
        
        # Create background mask from the FIRST band
        if idx == 0:
            background_mask = (band_array == 0)  # True = background
        
        # Multiply only valid data
        trait_map += band_array * coef
    
    except Exception as e:
        print(f"Error processing {band_name}: {str(e)}")
        continue

# Add intercept
trait_map += intercept

# Mask background as NaN
trait_map = np.where(background_mask, 0, trait_map)

# Write output GeoTIFF
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(output_tif, cols, rows, 1, gdal.GDT_Float32)
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)
out_ds.GetRasterBand(1).WriteArray(trait_map)
out_ds.FlushCache()
out_ds = None

print(f"Successfully created trait map: {output_tif}")
print(f"Trait map stats - Min: {np.nanmin(trait_map):.2f}, Max: {np.nanmax(trait_map):.2f}, Mean: {np.nanmean(trait_map):.2f}")


Successfully created trait map: D:\wenqu\chapter1_2\chapter2\PFT_ln_cover\No_scale\site2c_lichen.tif
Trait map stats - Min: -14.46, Max: 11.68, Mean: -0.86


# Forbs

# scale

In [6]:
output_tif = r'D:\wenqu\chapter1_2\chapter2\PFT_ln_cover\No_scale\site2c_forbs.tif'


# # Your list of selected bands
band_names = ['b1_std', 'b6_mean', 'b35_std', 'b25_std', 'b43_mean', 'b49_mean',
       'b43_std', 'b14_mean', 'b116_mean', 'b106_mean', 'b58_mean',
       'b121_mean', 'b21_std', 'b20_std', 'b31_std', 'b110_std',
       'b112_mean', 'b50_mean', 'b32_mean', 'b98_std', 'b74_std',
       'b46_std', 'b105_mean', 'b109_mean', 'b19_std', 'b110_mean',
       'b12_mean', 'b60_mean', 'b25_mean', 'b16_mean', 'b55_std',
       'b102_std', 'b51_mean', 'b13_std', 'b15_mean', 'b120_mean',
       'b117_std', 'b18_std', 'b9_mean', 'b59_mean', 'b56_std',
       'b109_std', 'b120_std', 'b4_mean', 'b8_std', 'b16_std', 'b2_mean',
       'b29_std', 'b17_std', 'b116_std']

plsr_coefficients  = np.array([-35.015938815643146,
 346.74172038650795,
 152.0952445727968,
 -36.179688729514226,
 174.68248258510215,
 135.63175513676654,
 -465.3031911397175,
 -150.18530401979314,
 -13.801798592582966,
 19.662202311685352,
 -130.16502363704794,
 -9.75842841369832,
 -289.9104326480918,
 -198.8710231980568,
 156.24704488534644,
 54.574787184033696,
 8.233422424120558,
 138.6268577703679,
 -124.67372540561425,
 23.399895008688326,
 -109.10057123208064,
 -406.1576409038045,
 14.353691429835566,
 5.270262963818011,
 -597.4559692112152,
 7.393112367175068,
 -49.58025998686548,
 -78.51017974396464,
 -261.15917663700327,
 125.38788367231547,
 211.32986156865292,
 8.049911784421843,
 178.70549259251928,
 682.6756944344443,
 182.41174484663796,
 -23.44632833000934,
 35.53297538169928,
 -329.44448407096087,
 -475.7634507974957,
 -116.10011552693713,
 231.97874248205298,
 27.638737766711564,
 7.621181693885809,
 -108.02766275852315,
 460.72393865902296,
 -519.877002255415,
 38.14021148432033,
 426.34225533335984,
 895.7713796785561,
 -55.908965727063624])


intercept = -5.60160065
trait_map = None
# Loop through bands and apply coefficients
for band_name, coef in zip(band_names, plsr_coefficients):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    if not os.path.exists(band_path):
        print(f"Missing file: {band_path}")
        continue
    band_array = np.load(band_path)

    if trait_map is None:
        trait_map = np.zeros_like(band_array)

    trait_map += band_array * coef

# Add intercept if available
trait_map += intercept

In [7]:
# Load reference image for size and geo info
ref_ds = gdal.Open(reference_tif)
geotransform = ref_ds.GetGeoTransform()
projection = ref_ds.GetProjection()
cols = ref_ds.RasterXSize
rows = ref_ds.RasterYSize
ref_ds = None

# Initialize trait map and background mask
trait_map = np.zeros((rows, cols), dtype=np.float32)
background_mask = None

# Process each band
for idx, (band_name, coef) in enumerate(zip(band_names, plsr_coefficients)):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    
    if not os.path.exists(band_path):
        print(f"Warning: Missing band file {band_path}")
        continue
    
    try:
        band_array = np.load(band_path)
        
        # Check dimension match
        if band_array.shape != (rows, cols):
            print(f"Warning: {band_name} has wrong dimensions {band_array.shape}, expected {(rows, cols)}")
            continue
        
        # Create background mask from the FIRST band
        if idx == 0:
            background_mask = (band_array == 0)  # True = background
        
        # Multiply only valid data
        trait_map += band_array * coef
    
    except Exception as e:
        print(f"Error processing {band_name}: {str(e)}")
        continue

# Add intercept
trait_map += intercept

# Mask background as NaN
trait_map = np.where(background_mask, 0, trait_map)

# Write output GeoTIFF
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(output_tif, cols, rows, 1, gdal.GDT_Float32)
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)
out_ds.GetRasterBand(1).WriteArray(trait_map)
out_ds.FlushCache()
out_ds = None

print(f"Successfully created trait map: {output_tif}")
print(f"Trait map stats - Min: {np.nanmin(trait_map):.2f}, Max: {np.nanmax(trait_map):.2f}, Mean: {np.nanmean(trait_map):.2f}")

Successfully created trait map: D:\wenqu\chapter1_2\chapter2\PFT_ln_cover\No_scale\site2c_forbs.tif
Trait map stats - Min: -21.58, Max: 15.81, Mean: -1.86


# evergreen_shrub

# scale

In [8]:
output_tif = r'D:\wenqu\chapter1_2\chapter2\PFT_ln_cover\No_scale\site2c_evergreen_shrub.tif'


# Your list of selected bands
band_names = ['b62_mean', 'b49_std', 'b25_mean', 'b24_std', 'b36_std', 'b18_std',
       'b57_std', 'b117_std', 'b66_std', 'b60_mean', 'b22_mean',
       'b13_std', 'b24_mean', 'b32_std', 'b113_mean', 'b56_mean',
       'b114_mean', 'b7_mean', 'b62_std', 'b28_std', 'b110_mean',
       'b112_std', 'b121_std', 'b112_mean', 'b8_mean', 'b122_mean',
       'b65_std', 'b59_std', 'b12_std', 'b109_mean', 'b118_std',
       'b115_std', 'b122_std', 'b63_std', 'b64_std', 'b10_std', 'b27_std',
       'b4_mean', 'b21_mean', 'b25_std', 'b120_mean', 'b1_mean',
       'b9_mean', 'b1_std', 'b113_std', 'b7_std', 'b9_std', 'b5_std',
       'b110_std', 'b8_std', 'b4_std']

plsr_coefficients  = np.array([16.88027063512285,
 -53.872362536076885,
 20.894579194401192,
 70.58948735689262,
 -74.5564458368106,
 42.45451826724658,
 57.901484611642466,
 -6.842929295588524,
 -9.575912333391686,
 14.303208414652188,
 24.380982029286752,
 -19.70925541407867,
 10.280294795447485,
 -58.590635118733495,
 4.036900366116312,
 -52.51480818233727,
 -3.26533732232425,
 0.28129599028500823,
 -39.15068977373119,
 59.338681218175054,
 -2.0684650087852523,
 23.357509851847,
 13.463084071549238,
 -3.003587181151228,
 -112.73267861147006,
 2.5948015835130054,
 -20.29288812238684,
 66.95349728758843,
 83.43123589570835,
 -9.137304983287219,
 4.410010780522211,
 13.899490015159195,
 18.008664623208062,
 -46.16917182944565,
 -39.1282302301451,
 -106.45741847930769,
 107.19332617422016,
 111.044405801203,
 70.25445544614422,
 122.1654753715427,
 8.84959346925595,
 -64.86992524404637,
 167.4879362908205,
 -20.511661603126992,
 -21.9247052351848,
 -190.3589475617852,
 -135.58108921273896,
 -120.5739501701664,
 30.152476917074097,
 -183.7020251771802,
 197.50471298015518])

intercept = -8.36722551
trait_map = None

# Loop through bands and apply coefficients
for band_name, coef in zip(band_names, plsr_coefficients):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    if not os.path.exists(band_path):
        print(f"Missing file: {band_path}")
        continue
    band_array = np.load(band_path)

    if trait_map is None:
        trait_map = np.zeros_like(band_array)

    trait_map += band_array * coef

# Add intercept if available
trait_map += intercept


In [9]:
# Load reference image for size and geo info
ref_ds = gdal.Open(reference_tif)
geotransform = ref_ds.GetGeoTransform()
projection = ref_ds.GetProjection()
cols = ref_ds.RasterXSize
rows = ref_ds.RasterYSize
ref_ds = None

# Initialize trait map and background mask
trait_map = np.zeros((rows, cols), dtype=np.float32)
background_mask = None

# Process each band
for idx, (band_name, coef) in enumerate(zip(band_names, plsr_coefficients)):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    
    if not os.path.exists(band_path):
        print(f"Warning: Missing band file {band_path}")
        continue
    
    try:
        band_array = np.load(band_path)
        
        # Check dimension match
        if band_array.shape != (rows, cols):
            print(f"Warning: {band_name} has wrong dimensions {band_array.shape}, expected {(rows, cols)}")
            continue
        
        # Create background mask from the FIRST band
        if idx == 0:
            background_mask = (band_array == 0)  # True = background
        
        # Multiply only valid data
        trait_map += band_array * coef
    
    except Exception as e:
        print(f"Error processing {band_name}: {str(e)}")
        continue

# Add intercept
trait_map += intercept

# Mask background as NaN
trait_map = np.where(background_mask, 0, trait_map)

# Write output GeoTIFF
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(output_tif, cols, rows, 1, gdal.GDT_Float32)
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)
out_ds.GetRasterBand(1).WriteArray(trait_map)
out_ds.FlushCache()
out_ds = None

print(f"Successfully created trait map: {output_tif}")
print(f"Trait map stats - Min: {np.nanmin(trait_map):.2f}, Max: {np.nanmax(trait_map):.2f}, Mean: {np.nanmean(trait_map):.2f}")

Successfully created trait map: D:\wenqu\chapter1_2\chapter2\PFT_ln_cover\No_scale\site2c_evergreen_shrub.tif
Trait map stats - Min: -14.01, Max: 13.62, Mean: -2.02


# deciduous_shrub

# scale

In [10]:
output_tif = r'D:\wenqu\chapter1_2\chapter2\PFT_ln_cover\No_scale\site2c_deciduous_shrub.tif'


# Your list of selected bands
band_names = ['b10_std', 'b85_mean', 'b33_mean', 'b112_mean', 'b2_mean',
       'b90_std', 'b88_std', 'b37_mean', 'b110_mean', 'b14_mean',
       'b98_mean', 'b102_std', 'b21_mean', 'b9_std', 'b19_mean',
       'b74_mean', 'b95_std', 'b118_mean', 'b38_std', 'b107_std',
       'b82_std', 'b15_mean', 'b32_std', 'b97_std', 'b114_mean',
       'b42_std', 'b28_mean', 'b13_mean', 'b24_std', 'b3_std', 'b7_mean',
       'b8_std', 'b94_std', 'b19_std', 'b15_std', 'b113_std', 'b116_mean',
       'b22_std', 'b113_mean', 'b13_std', 'b16_std', 'b37_std',
       'b119_std', 'b25_std', 'b59_std', 'b12_std', 'b120_std',
       'b100_std', 'b108_mean', 'b14_std', 'b34_std', 'b118_std',
       'b6_mean', 'b98_std', 'b17_std', 'b20_mean', 'b8_mean']

plsr_coefficients  = np.array([-44.40925245726402,
 8.66998451633172,
 51.09086872014375,
 -7.372060450841682,
 -13.643013868383996,
 -2.814357184972488,
 7.198418195222103,
 -38.39711857546021,
 2.5577585297312795,
 -62.98421636955575,
 -0.17737768857682815,
 32.9643553514045,
 -27.041395885408267,
 -34.859346328922925,
 38.555915449695696,
 -7.187072353864857,
 -23.65954278004468,
 2.9893290850750005,
 -23.554366863870897,
 -0.0013927793192673758,
 25.939356638464833,
 31.620062951563128,
 4.336453637093416,
 -1.103358907598885,
 -4.184778773815189,
 38.20437704776617,
 -46.453893717440906,
 13.118868737363085,
 22.767032676854267,
 16.763991167320643,
 -6.57640547987945,
 -25.991501952184105,
 13.937106749379934,
 -61.34132359395433,
 117.73714060152814,
 -0.11772157944068212,
 -10.140848864864646,
 -20.608078087397647,
 11.38848596361039,
 -3.8175710845664987,
 -21.758877633913546,
 40.194537270743304,
 -4.171960753297672,
 114.00684936058254,
 -9.025329076423112,
 -5.360710733361586,
 12.843553201056027,
 -24.363103514282344,
 3.7258828372711,
 40.770303421267315,
 -81.15811198593947,
 6.983110129398678,
 88.17000462695871,
 -37.53162904984354,
 -86.93415954853462,
 49.66617890661558,
 -74.3728432421292])

intercept = 0.04886115
trait_map = None

# Loop through bands and apply coefficients
for band_name, coef in zip(band_names, plsr_coefficients):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    if not os.path.exists(band_path):
        print(f"Missing file: {band_path}")
        continue
    band_array = np.load(band_path)

    if trait_map is None:
        trait_map = np.zeros_like(band_array)

    trait_map += band_array * coef

# Add intercept if available
trait_map += intercept

# Initialize trait map
trait_map = None
first_band = True

# Load reference image for size and geo info
ref_ds = gdal.Open(reference_tif)
geotransform = ref_ds.GetGeoTransform()
projection = ref_ds.GetProjection()
cols = ref_ds.RasterXSize
rows = ref_ds.RasterYSize
ref_ds = None

# Initialize trait map and background mask
trait_map = np.zeros((rows, cols), dtype=np.float32)
background_mask = None

# Process each band
for idx, (band_name, coef) in enumerate(zip(band_names, plsr_coefficients)):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    
    if not os.path.exists(band_path):
        print(f"Warning: Missing band file {band_path}")
        continue
    
    try:
        band_array = np.load(band_path)
        
        # Check dimension match
        if band_array.shape != (rows, cols):
            print(f"Warning: {band_name} has wrong dimensions {band_array.shape}, expected {(rows, cols)}")
            continue
        
        # Create background mask from the FIRST band
        if idx == 0:
            background_mask = (band_array == 0)  # True = background
        
        # Multiply only valid data
        trait_map += band_array * coef
    
    except Exception as e:
        print(f"Error processing {band_name}: {str(e)}")
        continue

# Add intercept
trait_map += intercept

# Mask background as NaN
trait_map = np.where(background_mask, 0, trait_map)

# Write output GeoTIFF
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(output_tif, cols, rows, 1, gdal.GDT_Float32)
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)
out_ds.GetRasterBand(1).WriteArray(trait_map)
out_ds.FlushCache()
out_ds = None

print(f"Successfully created trait map: {output_tif}")
print(f"Trait map stats - Min: {np.nanmin(trait_map):.2f}, Max: {np.nanmax(trait_map):.2f}, Mean: {np.nanmean(trait_map):.2f}")

Successfully created trait map: D:\wenqu\chapter1_2\chapter2\PFT_ln_cover\No_scale\site2c_deciduous_shrub.tif
Trait map stats - Min: -1.66, Max: 1.95, Mean: 0.06


# non-veg

In [11]:
output_tif = r'D:\wenqu\chapter1_2\chapter2\pft_spectra_origianl\PFT_Scale\site2a_non_veg.tif'


# Your list of selected bands
band_names = ['b40_std', 'b10_mean', 'b69_std', 'b66_mean', 'b29_std',
       'b61_mean', 'b103_mean', 'b44_std', 'b89_std', 'b121_mean',
       'b70_std', 'b100_std', 'b72_std', 'b119_mean', 'b41_std',
       'b34_mean', 'b74_mean', 'b10_std', 'b58_std', 'b65_std', 'b88_std',
       'b29_mean', 'b30_mean', 'b102_std', 'b9_mean', 'b49_std',
       'b74_std', 'b71_std', 'b57_std', 'b98_mean', 'b63_std', 'b14_mean',
       'b37_mean', 'b22_mean', 'b117_std', 'b106_std', 'b52_std',
       'b12_std', 'b64_std', 'b12_mean', 'b45_std', 'b13_mean', 'b8_mean',
       'b92_std', 'b25_mean', 'b62_mean', 'b73_std', 'b99_std', 'b86_std',
       'b122_std', 'b105_std', 'b95_std', 'b63_mean', 'b118_mean',
       'b54_std', 'b2_mean', 'b34_std', 'b114_mean', 'b16_mean',
       'b13_std', 'b64_mean', 'b115_mean', 'b60_std', 'b56_std',
       'b104_std', 'b111_std', 'b59_std', 'b55_std', 'b61_std', 'b14_std',
       'b109_std', 'b26_std', 'b21_std', 'b16_std', 'b113_mean',
       'b94_std', 'b15_mean', 'b31_std', 'b121_std', 'b114_std',
       'b93_std', 'b4_mean', 'b1_std', 'b103_std', 'b110_std', 'b120_std',
       'b107_std', 'b17_std']

plsr_coefficients  = np.array([-0.06658989,  0.067972  , -0.06682704,  0.04242981,  0.03442325,
        -0.01632595,  0.03921355, -0.1096973 ,  0.01719706,  0.107968  ,
        -0.06758719,  0.03689845, -0.05113756,  0.02682328,  0.0032035 ,
         0.01954775, -0.12970487,  0.039819  ,  0.01410427,  0.0410501 ,
         0.03338663, -0.11502411, -0.11353046,  0.07323935, -0.0687682 ,
        -0.02574367, -0.06835562, -0.07878877,  0.04783218, -0.09861014,
         0.04648375,  0.0729008 , -0.09153566, -0.05267703, -0.05569812,
         0.01356955,  0.01258928, -0.0185756 ,  0.05987293, -0.0715106 ,
        -0.07832922, -0.04353424, -0.0066255 ,  0.06589166, -0.05727098,
         0.06238873, -0.12093485,  0.00753768,  0.01243434, -0.02656059,
         0.06941608,  0.07548251,  0.10605639, -0.12779367,  0.11601894,
        -0.04752536, -0.02192481,  0.03510116,  0.05964066, -0.02190204,
         0.10276472,  0.02522572, -0.07937021,  0.07242103,  0.10827401,
         0.08401527, -0.12532569,  0.11878407, -0.04814257,  0.06387707,
         0.07594613, -0.06846548, -0.03846321,  0.14674765,  0.10620917,
         0.10651083,  0.04561171,  0.14839699, -0.05824089, -0.06134647,
         0.1273556 ,  0.14070105, -0.11577339,  0.15040272, -0.1588453 ,
        -0.17756072, -0.16933456, -0.16607823])

intercept = 0.13818146
trait_map = None

# Loop through bands and apply coefficients
for band_name, coef in zip(band_names, plsr_coefficients):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    if not os.path.exists(band_path):
        print(f"Missing file: {band_path}")
        continue
    band_array = np.load(band_path)

    if trait_map is None:
        trait_map = np.zeros_like(band_array)

    trait_map += band_array * coef

# Add intercept if available
trait_map += intercept

# Initialize trait map
trait_map = None
first_band = True

# Load reference image for size and geo info
ref_ds = gdal.Open(reference_tif)
geotransform = ref_ds.GetGeoTransform()
projection = ref_ds.GetProjection()
cols = ref_ds.RasterXSize
rows = ref_ds.RasterYSize
ref_ds = None

# Initialize trait map and background mask
trait_map = np.zeros((rows, cols), dtype=np.float32)
background_mask = None

# Process each band
for idx, (band_name, coef) in enumerate(zip(band_names, plsr_coefficients)):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    
    if not os.path.exists(band_path):
        print(f"Warning: Missing band file {band_path}")
        continue
    
    try:
        band_array = np.load(band_path)
        
        # Check dimension match
        if band_array.shape != (rows, cols):
            print(f"Warning: {band_name} has wrong dimensions {band_array.shape}, expected {(rows, cols)}")
            continue
        
        # Create background mask from the FIRST band
        if idx == 0:
            background_mask = (band_array == 0)  # True = background
        
        # Multiply only valid data
        trait_map += band_array * coef
    
    except Exception as e:
        print(f"Error processing {band_name}: {str(e)}")
        continue

# Add intercept
trait_map += intercept

# Mask background as NaN
trait_map = np.where(background_mask, 0, trait_map)

# Write output GeoTIFF
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(output_tif, cols, rows, 1, gdal.GDT_Float32)
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)
out_ds.GetRasterBand(1).WriteArray(trait_map)
out_ds.FlushCache()
out_ds = None

print(f"Successfully created trait map: {output_tif}")
print(f"Trait map stats - Min: {np.nanmin(trait_map):.2f}, Max: {np.nanmax(trait_map):.2f}, Mean: {np.nanmean(trait_map):.2f}")

Successfully created trait map: D:\wenqu\chapter1_2\chapter2\pft_spectra_origianl\PFT_Scale\site2c_non_veg.tif
Trait map stats - Min: 0.00, Max: 0.20, Mean: 0.07


# d13c

# scale

In [4]:
output_tif = r'D:\wenqu\chapter1_2\uas_trait_map\site6_d13c.tif'


# Your list of selected bands
band_names = ['b82_mean', 'b56_mean', 'b75_mean', 'b67_mean', 'b99_mean',
       'b97_mean', 'b57_mean', 'b35_mean', 'b26_mean', 'b5_mean',
       'b116_mean', 'b17_mean', 'b1_mean', 'b14_mean', 'b73_mean',
       'b28_mean', 'b3_mean', 'b39_mean', 'b51_mean', 'b122_mean',
       'b40_mean', 'b30_mean', 'b49_mean', 'b42_mean', 'b76_mean',
       'b48_mean', 'b8_mean', 'b34_mean', 'b121_mean', 'b52_mean',
       'b15_mean', 'b7_mean']

plsr_coefficients  = np.array([236.61852623486863,
 -482.88222925347776,
 138.77216745347565,
 -44.673199569661506,
 -130.83854381839146,
 -69.32322347897814,
 -130.99011762271542,
 174.92483319157319,
 86.31135633974849,
 103.09103836913387,
 44.41173006786578,
 -0.13247107610119974,
 83.19520249834109,
 525.2716626984129,
 78.22438490255212,
 290.4246751239607,
 -97.49652904815359,
 -232.4310991099464,
 161.20342046086108,
 104.632806150325,
 -387.46969869396503,
 116.29419474030463,
 138.606982935393,
 -116.52149104887903,
 -357.40088662930526,
 -128.67990761302707,
 -366.92702046262247,
 -364.4010164924634,
 -70.19222226778966,
 765.1253660579733,
 458.31738032251434,
 -476.77803435807124])

intercept = -27.40784303
trait_map = None

# Initialize trait map
trait_map = None
first_band = True

# Load reference image for size and geo info
ref_ds = gdal.Open(reference_tif)
geotransform = ref_ds.GetGeoTransform()
projection = ref_ds.GetProjection()
cols = ref_ds.RasterXSize
rows = ref_ds.RasterYSize
ref_ds = None

# Initialize trait map and background mask
trait_map = np.zeros((rows, cols), dtype=np.float32)
background_mask = None

# Process each band
for idx, (band_name, coef) in enumerate(zip(band_names, plsr_coefficients)):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    
    if not os.path.exists(band_path):
        print(f"Warning: Missing band file {band_path}")
        continue
    
    try:
        band_array = np.load(band_path)
        
        # Check dimension match
        if band_array.shape != (rows, cols):
            print(f"Warning: {band_name} has wrong dimensions {band_array.shape}, expected {(rows, cols)}")
            continue
        
        # Create background mask from the FIRST band
        if idx == 0:
            background_mask = (band_array == 0)  # True = background
        
        # Multiply only valid data
        trait_map += band_array * coef
    
    except Exception as e:
        print(f"Error processing {band_name}: {str(e)}")
        continue

# Add intercept
trait_map += intercept

# Mask background as NaN
trait_map = np.where(background_mask, 0, trait_map)

# Write output GeoTIFF
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(output_tif, cols, rows, 1, gdal.GDT_Float32)
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)
out_ds.GetRasterBand(1).WriteArray(trait_map)
out_ds.FlushCache()
out_ds = None

print(f"Successfully created trait map: {output_tif}")
print(f"Trait map stats - Min: {np.nanmin(trait_map):.2f}, Max: {np.nanmax(trait_map):.2f}, Mean: {np.nanmean(trait_map):.2f}")

AttributeError: 'NoneType' object has no attribute 'SetGeoTransform'

# d15n

In [48]:
output_tif = r'D:\wenqu\chapter1_2\uas_trait_map\site2c_d15n.tif'


# PLSR model parameters
band_names = ['b74_mean', 'b5_mean', 'b57_mean', 'b58_mean', 'b38_std',
             'b60_mean', 'b25_std', 'b95_std', 'b30_std', 'b9_mean', 'b114_std',
             'b91_std', 'b23_std', 'b18_std', 'b111_mean', 'b14_mean',
             'b101_std', 'b7_mean', 'b122_std', 'b59_std', 'b107_mean',
             'b118_mean', 'b122_mean', 'b13_std', 'b10_std', 'b98_std',
             'b3_mean', 'b99_std', 'b15_std', 'b109_std', 'b14_std', 'b9_std',
             'b8_mean', 'b108_mean', 'b113_std', 'b118_std']

plsr_coefficients = np.array([61.95069832038617,
 -443.8175861956897,
 244.61697666292002,
 161.76757240677037,
 890.6348221654105,
 -179.8869698051848,
 -510.1111454297827,
 149.63785994193933,
 -353.6891690306521,
 -537.7722892932843,
 111.57768804560136,
 -24.366765168206275,
 262.9394749098357,
 -726.5451610814567,
 -22.324825795701166,
 -640.9355498468841,
 -41.70931448381327,
 1007.2321141650461,
 77.62091383540057,
 -535.1730005567188,
 64.55454750642573,
 -69.0054026100846,
 -48.2102651040233,
 282.5904906929578,
 -1322.7513820446086,
 16.7844528529304,
 -157.0443501507085,
 -279.05763846899987,
 -584.5370647143357,
 199.05976457311684,
 1960.7291117127768,
 1359.530026185485,
 616.2063977631504,
 30.11636147145316,
 -182.94787076813563,
 -23.993009150288234])

intercept = -4.16656468

# Initialize trait map
trait_map = None
first_band = True

# Load reference image for size and geo info
ref_ds = gdal.Open(reference_tif)
geotransform = ref_ds.GetGeoTransform()
projection = ref_ds.GetProjection()
cols = ref_ds.RasterXSize
rows = ref_ds.RasterYSize
ref_ds = None

# Initialize trait map and background mask
trait_map = np.zeros((rows, cols), dtype=np.float32)
background_mask = None

# Process each band
for idx, (band_name, coef) in enumerate(zip(band_names, plsr_coefficients)):
    band_path = os.path.join(npy_folder, f"{band_name}.npy")
    
    if not os.path.exists(band_path):
        print(f"Warning: Missing band file {band_path}")
        continue
    
    try:
        band_array = np.load(band_path)
        
        # Check dimension match
        if band_array.shape != (rows, cols):
            print(f"Warning: {band_name} has wrong dimensions {band_array.shape}, expected {(rows, cols)}")
            continue
        
        # Create background mask from the FIRST band
        if idx == 0:
            background_mask = (band_array == 0)  # True = background
        
        # Multiply only valid data
        trait_map += band_array * coef
    
    except Exception as e:
        print(f"Error processing {band_name}: {str(e)}")
        continue

# Add intercept
trait_map += intercept

# Mask background as NaN
trait_map = np.where(background_mask, 0, trait_map)

# Write output GeoTIFF
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(output_tif, cols, rows, 1, gdal.GDT_Float32)
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)
out_ds.GetRasterBand(1).WriteArray(trait_map)
out_ds.FlushCache()
out_ds = None

print(f"Successfully created trait map: {output_tif}")
print(f"Trait map stats - Min: {np.nanmin(trait_map):.2f}, Max: {np.nanmax(trait_map):.2f}, Mean: {np.nanmean(trait_map):.2f}")

Successfully created trait map: D:\wenqu\chapter1_2\uas_trait_map\site2c_d15n.tif
Trait map stats - Min: -23.05, Max: 22.20, Mean: 0.40
